In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/riteshkumarweb/insurence-1000-dataset/insurance_1000_records.csv


In [2]:
df = pd.read_csv('/kaggle/input/datasets/riteshkumarweb/insurence-1000-dataset/insurance_1000_records.csv')

In [3]:
df

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
0,58,100.6,1.51,8.101864,True,Chennai,unemployed,High
1,65,51.5,1.90,13.235733,True,Bangalore,freelancer,High
2,32,47.0,1.70,6.766292,False,Jaipur,government_job,Low
3,55,110.2,1.59,23.005414,True,Indore,student,High
4,35,116.8,1.53,10.761242,True,Kota,retired,High
...,...,...,...,...,...,...,...,...
995,25,48.2,1.90,9.747728,True,Chennai,private_job,Medium
996,25,60.4,1.91,27.134071,True,Mysore,retired,Low
997,23,101.0,1.52,9.043894,False,Mysore,freelancer,Medium
998,72,99.6,1.70,8.012853,True,Chennai,private_job,High


In [4]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
636,32,89.2,1.90,25.857760,True,Surat,business_owner,Low
520,51,106.6,1.88,15.942821,True,Bangalore,student,High
123,39,52.8,1.61,5.663921,False,Nagpur,student,Low
941,47,101.0,1.88,1.707991,False,Surat,private_job,Medium
506,42,76.4,1.86,16.529918,False,Jaipur,government_job,Low


In [5]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

In [6]:
df['occupation'].unique()

array(['unemployed', 'freelancer', 'government_job', 'student', 'retired',
       'private_job', 'business_owner'], dtype=object)

In [7]:
df_feat = df.copy()

In [8]:
# Feature 1: BMI
df_feat["bmi"] = df_feat["weight"] / (df_feat["height"] ** 2)

In [9]:
# Feature 2: Age Group
def age_group(age):
    if age < 25:
        return "young"
    elif age < 45:
        return "adult"
    elif age < 60:
        return "middle_aged"
    return "senior"

In [10]:
df_feat["age_group"] = df_feat["age"].apply(age_group)

In [11]:
# Feature 3: Lifestyle Risk
def lifestyle_risk(row):
    if row["smoker"] and row["bmi"] > 30:
        return "high"
    elif row["smoker"] or row["bmi"] > 27:
        return "medium"
    else:
        return "low"
     

In [12]:
df_feat["lifestyle_risk"] = df_feat.apply(lifestyle_risk, axis=1)

In [13]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

In [14]:

# Feature 4: City Tier
def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    else:
        return 3

In [15]:
df_feat["city_tier"] = df_feat["city"].apply(city_tier)
     


In [16]:
df_feat.drop(columns=['age', 'weight', 'height', 'smoker', 'city'])[['income_lpa', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'insurance_premium_category']].sample(5)
     

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
462,3.622206,freelancer,38.584184,adult,medium,1,Medium
50,21.039569,private_job,50.831104,young,high,1,Medium
711,19.300378,student,37.864733,middle_aged,medium,2,Medium
672,16.014538,business_owner,32.690542,senior,high,1,High
590,21.300414,unemployed,32.894737,senior,medium,2,Medium


In [17]:
# Select features and target
X = df_feat[["bmi", "age_group", "lifestyle_risk", "city_tier", "income_lpa", "occupation"]]
y = df_feat["insurance_premium_category"]

In [18]:
X

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
0,44.120872,middle_aged,high,1,8.101864,unemployed
1,14.265928,senior,medium,1,13.235733,freelancer
2,16.262976,adult,low,2,6.766292,government_job
3,43.590048,middle_aged,high,2,23.005414,student
4,49.895339,adult,high,3,10.761242,retired
...,...,...,...,...,...,...
995,13.351801,adult,medium,1,9.747728,private_job
996,16.556564,adult,medium,2,27.134071,retired
997,43.715374,young,medium,2,9.043894,freelancer
998,34.463668,senior,high,1,8.012853,private_job


In [19]:
y

0        High
1        High
2         Low
3        High
4        High
        ...  
995    Medium
996       Low
997    Medium
998      High
999       Low
Name: insurance_premium_category, Length: 1000, dtype: object

In [20]:
# Define categorical and numeric features
categorical_features = ["age_group", "lifestyle_risk", "occupation", "city_tier"]
numeric_features = ["bmi", "income_lpa"]

# Create column transformer for OHE
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

# Create a pipeline with preprocessing and random forest classifier
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

# Split data and train model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'occupation', 'city_tier']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'income_lpa'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [21]:
# Predict and evaluate
y_pred = pipeline.predict(X_test)
accuracy_score(y_test, y_pred)

0.86

In [22]:

X_test.sample(5)

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
479,38.733074,middle_aged,high,2,11.404711,business_owner
507,50.763479,adult,high,1,21.714865,private_job
473,33.451435,adult,high,3,8.225285,private_job
911,37.653905,middle_aged,high,1,24.716205,government_job
685,43.362525,middle_aged,medium,2,26.548630,freelancer


In [23]:
import pickle

# Save the trained pipeline using pickle
pickle_model_path = "model.pkl"
with open(pickle_model_path, "wb") as f:
    pickle.dump(pipeline, f)

In [24]:
import sklearn
print(sklearn.__version__)

1.6.1


In [25]:
import platform
print(platform.python_version())

3.12.13


In [26]:
import sys
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [27]:
import sklearn
import pandas
import numpy
import sys

print("Python:", sys.version)
print("Scikit-Learn:", sklearn.__version__)
print("Pandas:", pandas.__version__)
print("NumPy:", numpy.__version__)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Scikit-Learn: 1.6.1
Pandas: 2.3.3
NumPy: 2.4.6


In [28]:
!pip install scikit-learn